# B3b Defense – 07 LLM Macroeconomic Analysis

**Objective:** Send the XGBoost forecast results (`defense_forecast_2026_sac.csv`) and
the model driver breakdown (`defense_forecast_2026_drivers_sac.csv`) to Claude Sonnet 4.6
and request a macroeconomic interpretation tailored to a company planning to enter the
US defense segment with new products in 2026.

**Target variable:** FDEFX — Federal Government National Defense Consumption Expenditures
and Gross Investment (USD billions, SAAR). This is the primary forecast output; ADEFNO
(new defense orders) is a model input feature, not the forecasted quantity.

The LLM acts as a macroeconomic analyst writing for a corporate controller — it does not
re-run any model, but contextualises the quantitative outputs for strategic planning and
management communication.

In [5]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
import anthropic
from IPython.display import display, Markdown

# Force UTF-8 output on Windows (avoids CP1252 encoding errors)
if sys.stdout.encoding and sys.stdout.encoding.lower() != 'utf-8':
    sys.stdout.reconfigure(encoding='utf-8')

# Load API key from project .env
load_dotenv(dotenv_path='../../.env')

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
if not ANTHROPIC_API_KEY:
    raise EnvironmentError(
        'ANTHROPIC_API_KEY not found. Add it to the project .env file: '
        'ANTHROPIC_API_KEY=sk-ant-...'
    )

client         = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
DATA_RAW       = '../data/raw/'
DATA_PROCESSED = '../data/processed/'
OUTPUT_DIR     = Path('../output')
OUTPUT_DIR.mkdir(exist_ok=True)

print('Anthropic client initialised')

Anthropic client initialised


In [6]:
# ===========================================================================
# 1. Load FDEFX statistics — PRIMARY FORECAST TARGET
# ===========================================================================
fdefx_raw = pd.read_csv(DATA_RAW + 'FDEFX.csv', parse_dates=['observation_date'])
fdefx_raw = fdefx_raw.rename(columns={'observation_date': 'date'}).sort_values('date')

fdefx_hist     = fdefx_raw[fdefx_raw['date'] <= '2025-12-31'].copy()
fdefx_last_row = fdefx_hist.iloc[-1]
fdefx_last_val = fdefx_last_row['FDEFX']
fdefx_last_qtr = fdefx_last_row['date']

q_num = (fdefx_last_qtr.month - 1) // 3 + 1
fdefx_last_quarter = f'Q{q_num} {fdefx_last_qtr.year}'

prior_date      = fdefx_last_qtr - pd.DateOffset(years=1)
fdefx_prior_row = fdefx_hist[fdefx_hist['date'] == prior_date]
if fdefx_prior_row.empty:
    fdefx_prior_row = fdefx_hist.iloc[-5:-4]
fdefx_prior_val = float(fdefx_prior_row['FDEFX'].iloc[0])
fdefx_yoy       = (fdefx_last_val - fdefx_prior_val) / fdefx_prior_val * 100

# 5-year range for context
fdefx_2020_2025 = fdefx_hist[fdefx_hist['date'].dt.year >= 2020]['FDEFX']
fdefx_min_val   = fdefx_2020_2025.min()
fdefx_max_val   = fdefx_2020_2025.max()

# ===========================================================================
# 2. Load ADEFNO statistics — context feature (defense new orders)
# ===========================================================================
adefno_raw = pd.read_csv(DATA_RAW + 'ADEFNO.csv', parse_dates=['observation_date'])
adefno_raw = adefno_raw.rename(columns={'observation_date': 'date'}).sort_values('date')

adefno_hist       = adefno_raw[adefno_raw['date'] <= '2025-12-31'].copy()
last_row          = adefno_hist.iloc[-1]
adefno_last_month = last_row['date'].strftime('%b %Y')
adefno_last_value = last_row['ADEFNO']

adefno_2025    = adefno_hist[adefno_hist['date'].dt.year == 2025]['ADEFNO']
adefno_12m_avg = adefno_2025.mean()
adefno_vs_avg_pct = (adefno_last_value / adefno_12m_avg - 1) * 100

# ===========================================================================
# 3. Load IPB52300S statistics — defense production index (context feature)
# ===========================================================================
ipb_raw = pd.read_csv(DATA_RAW + 'IPB52300S.csv', parse_dates=['observation_date'])
ipb_raw = ipb_raw.rename(columns={'observation_date': 'date'}).sort_values('date')

ipb_hist             = ipb_raw[ipb_raw['date'] <= '2025-12-31'].copy()
ipb_last_row         = ipb_hist.iloc[-1]
ipb_last_val         = ipb_last_row['IPB52300S']
ipb_last_month_label = ipb_last_row['date'].strftime('%b %Y')

ipb_jan_val    = ipb_hist[ipb_hist['date'] == '2025-01-01']['IPB52300S'].iloc[0]
ipb_12m_change = ipb_last_val - ipb_jan_val
ipb_direction  = 'upward' if ipb_12m_change > 0 else 'downward'

# ===========================================================================
# 4. Load FDEFX forecast and compute summary statistics
# ===========================================================================
forecast_df = pd.read_csv(DATA_PROCESSED + 'defense_forecast_2026_sac.csv')
forecast_df['date_parsed'] = pd.to_datetime(
    forecast_df['Date'].astype(str), format='%Y%m'
)
forecast_df['Month']      = forecast_df['date_parsed'].dt.strftime('%b %Y')
# FDEFX is in USD bn SAAR — convert from full USD back to billions for display
forecast_df['FDEFX_USD_bn'] = (forecast_df['Net_Value_USD'] / 1e9).round(1)

forecast_avg_bn = round(forecast_df['Net_Value_USD'].mean() / 1e9, 1)
forecast_min_bn = round(forecast_df['Net_Value_USD'].min() / 1e9, 1)
forecast_max_bn = round(forecast_df['Net_Value_USD'].max() / 1e9, 1)
fdefx_step_down = round(fdefx_last_val - forecast_avg_bn, 1)

jan_2026_val = round(
    forecast_df[forecast_df['date_parsed'].dt.month == 1]['Net_Value_USD'].values[0] / 1e9, 1
)

forecast_table = forecast_df[['Month', 'FDEFX_USD_bn']].to_string(index=False)

# ===========================================================================
# 5. Load model driver breakdown (SHAP)
# ===========================================================================
drivers_df   = pd.read_csv(DATA_PROCESSED + 'defense_forecast_2026_drivers_sac.csv')
feature_shap = drivers_df[~drivers_df['Driver'].isin(['base_value', 'prediction'])].copy()
feature_shap['abs_shap'] = feature_shap['SHAP_Value_USD_bn'].abs()

base_value_bn = float(
    drivers_df[drivers_df['Driver'] == 'base_value']['SHAP_Value_USD_bn'].iloc[0]
)

importance_df = (
    feature_shap.groupby('Driver')['abs_shap']
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
    .rename(columns={'Driver': 'Feature', 'abs_shap': 'Avg_contribution_USD_bn'})
)
importance_df['Avg_contribution_USD_bn'] = importance_df['Avg_contribution_USD_bn'].round(2)
importance_table = importance_df.to_string(index=False)

# Individual and cumulative driver contributions for prompt narrative
def _get_contrib(feature):
    row = importance_df[importance_df['Feature'] == feature]['Avg_contribution_USD_bn']
    return float(row.values[0]) if len(row) > 0 else 0.0

lag1_contrib       = _get_contrib('fdefx_lag_1')
lag2_contrib       = _get_contrib('fdefx_lag_2')
lag3_contrib       = _get_contrib('fdefx_lag_3')
ipb_contrib_val    = _get_contrib('IPB52300S')
adefno_contrib_val = _get_contrib('ADEFNO_lag_6')
cumul_lag123       = round(lag1_contrib + lag2_contrib + lag3_contrib, 2)
cumul_external     = round(ipb_contrib_val + adefno_contrib_val, 2)

print(f'FDEFX last quarter:    {fdefx_last_quarter}: {fdefx_last_val:,.1f} USD bn SAAR (YoY {fdefx_yoy:+.1f}%)')
print(f'FDEFX 2020–2025 range: {fdefx_min_val:,.1f} – {fdefx_max_val:,.1f} USD bn SAAR')
print(f'ADEFNO last value:     {adefno_last_month}: {adefno_last_value:,.0f} USD mn ({adefno_vs_avg_pct:+.1f}% vs 2025 avg)')
print(f'ADEFNO 12m avg 2025:   {adefno_12m_avg:,.0f} USD mn')
print(f'IPB52300S last:        {ipb_last_month_label}: {ipb_last_val:.2f} (12m change: {ipb_12m_change:+.2f}, {ipb_direction})')
print(f'Forecast 2026:         Jan {jan_2026_val} → avg {forecast_avg_bn} → range {forecast_min_bn}–{forecast_max_bn} USD bn SAAR')
print(f'Step-down from Q4 act: {fdefx_step_down:.1f} USD bn')
print(f'Model reference level: {base_value_bn:.1f} USD bn SAAR (SHAP base value)')
print(f'Cumul. lag-1/2/3:      {cumul_lag123:.2f} USD bn  |  External signals: {cumul_external:.2f} USD bn')

FDEFX last quarter:    Q4 2025: 1,159.2 USD bn SAAR (YoY +3.2%)
FDEFX 2020–2025 range: 872.0 – 1,161.9 USD bn SAAR
ADEFNO last value:     Dec 2025: 18,774 USD mn (+19.0% vs 2025 avg)
ADEFNO 12m avg 2025:   15,780 USD mn
IPB52300S last:        Dec 2025: 112.96 (12m change: +3.48, upward)
Forecast 2026:         Jan 906.6 → avg 901.1 → range 897.8–906.6 USD bn SAAR
Step-down from Q4 act: 258.1 USD bn
Model reference level: 733.5 USD bn SAAR (SHAP base value)
Cumul. lag-1/2/3:      122.46 USD bn  |  External signals: 6.42 USD bn


## Prompt Design

The system prompt positions Claude as a senior macroeconomic analyst writing for a
**corporate controller** — technical ML terms (XGBoost, SHAP, autoregressive) are avoided
in the output; business-language equivalents are used instead.

**Target variable:** FDEFX (realized US federal defense expenditures, USD bn SAAR).
ADEFNO (new defense orders) and IPB52300S (production index) are model input features,
not the forecasted quantity.

**Anti-hallucination rules injected into the user prompt:**
- Use **only** the numerical values explicitly provided (no memory-based statistics)
- Every factual claim not derivable from the tables must cite an **official primary source**
  with a verifiable URL (FRED, BEA, US Census, Federal Reserve, CBO, DoD Comptroller)
- No secondary sources; no industry association claims without a direct primary URL
- If a claim cannot be supported → **omit it entirely**
- Scenario probabilities flagged as **illustrative** unless sourced

**Output structure (4 sections + exec summary):**
1. Defense Market Volume & Outlook (FDEFX trajectory, step-down from 2025, stabilisation)
2. What Drives the Forecast (plain-language: recent spending momentum, policy signal, production capacity)
3. Quarterly Monitoring KPIs (Markdown table: 4 KPIs with source URL, cadence, what to watch)
4. Key Risks & Forecast Limitations (US budget dynamics, frozen macro inputs)
5. Executive Summary (exactly 6 bullets, max 2 lines each — for SAC Planning Story)

In [7]:
# ---------------------------------------------------------------------------
# Model selection — uncomment the model you want to use
# ---------------------------------------------------------------------------
# claude-sonnet-4-5   $3.00 input / $15.00 output per 1M tokens
# MODEL = 'claude-sonnet-4-5'

# claude-haiku-4-5    $1.00 input / $5.00 output  per 1M tokens  (cheapest)
# MODEL = 'claude-haiku-4-5'

# claude-sonnet-4-6   $3.00 input / $15.00 output per 1M tokens  (latest)
MODEL = 'claude-sonnet-4-6'

# claude-opus-4-6     $5.00 input / $25.00 output per 1M tokens  (most capable)
# MODEL = 'claude-opus-4-6'
# ---------------------------------------------------------------------------

MAX_TOKENS = 3500

# ---------------------------------------------------------------------------
# Prompts
# ---------------------------------------------------------------------------
SYSTEM_PROMPT = """\
You are a senior macroeconomic analyst. Your audience is a corporate controller at an
industrial B2B company — not a data scientist. Write in clear business language.
Avoid technical modelling terms such as "XGBoost", "SHAP", "autoregressive", or
"time-series split". Instead use plain equivalents:
  - "the forecast model" (not "XGBoost")
  - "what the model relied on most" or "main forecast driver" (not "SHAP")
  - "recent spending history" (not "autoregressive lags")
  - "spending rate" or "annualised spending level" (not "SAAR")
  - "the model's reference level from training data" (not "long-run average" or "baseline")
Prioritize verifiable sourcing over narrative flourish. Brevity is a feature.

Be intellectually honest: if the data shows an inconvenient pattern (e.g., that a
supposedly important feature contributes little), surface it rather than smoothing
it over. A controller needs to know where the model is strong and where it is weak.\
"""

USER_PROMPT = f"""\
## Sourcing Rules — Read Before Answering

Use **only** the numerical values explicitly provided in this prompt.
Do not introduce any statistics, percentages, ratios, or historical figures from memory.

For every factual claim not directly derivable from the data tables below,
cite an **official primary source** with a specific, verifiable URL from this list:
  - FRED:            https://fred.stlouisfed.org
  - BEA:             https://www.bea.gov
  - US Census:       https://www.census.gov
  - Federal Reserve: https://www.federalreserve.gov
  - CBO:             https://www.cbo.gov
  - DoD Comptroller: https://comptroller.defense.gov

No secondary sources. No industry association claims without a direct primary URL.
If you cannot support a claim with either (a) the data provided or (b) an official
primary source with a verifiable URL, **omit the claim entirely**.
Flag any scenario probabilities or ranges as **illustrative** unless sourced.

---

## Company Context

An industrial B2B manufacturer (compressors and pneumatic tools, Germany-based) is
launching products in the US defense segment in 2026 — with zero prior defense revenue.

Revenue targets are set in SAP Analytics Cloud (SAC) using a three-block structure:
  - B1: Confirmed order backlog + weighted quotation pipeline (existing customers)
  - B2: New customer estimates
  - B3: Defense market volume × management market share assumption ÷ 12   ← this analysis

All three blocks must aggregate to a consistent invoice-equivalent revenue figure in SAC.
The forecast below targets FDEFX (realised federal defense expenditures) rather than
new defense orders (ADEFNO), precisely so that B3 is semantically consistent with the
invoice-based logic of B1 and B2. This choice is relevant context for Section 1.

The market share percentage is a separate management decision and is **not part of
this analysis**.

---

## Verified Input Data

### 1. FDEFX — US Federal Defense Consumption Expenditures and Gross Investment
Source: FRED series FDEFX (https://fred.stlouisfed.org/series/FDEFX)
Unit: USD billions, annualised spending rate (seasonally adjusted)
Interpretation: Each value represents the annual pace of US federal defense spending —
  e.g., a value of 901 means the US was spending defense funds at a rate of $901 bn/year
  in that period, not that $901 bn was spent in a single month.

This is the quantity the forecast model predicts. It measures realised expenditures
(money actually flowing from the federal government to defense suppliers), which is
the invoice-equivalent quantity — as opposed to new orders, which are commitments
that convert to expenditures only over time.

Last observed value:     {fdefx_last_quarter}: {fdefx_last_val:,.1f} USD bn
Prior-year same quarter: {fdefx_prior_val:,.1f} USD bn
Year-on-year change:     {fdefx_yoy:+.1f}%
2020–2025 range:         {fdefx_min_val:,.1f} – {fdefx_max_val:,.1f} USD bn

### 2. ADEFNO — US New Defense Orders (Capital Goods)
Source: FRED series ADEFNO (https://fred.stlouisfed.org/series/ADEFNO)
Unit: USD millions, seasonally adjusted
Interpretation: Orders placed by the US military with manufacturers — a leading signal
  for future defense activity. Available to the model as an input feature, alongside
  its own recent FDEFX history.

Last observed value:   {adefno_last_month}: {adefno_last_value:,.0f} USD mn
12-month average 2025: {adefno_12m_avg:,.0f} USD mn  (Jan–Dec 2025)
{adefno_last_month} vs 2025 avg: {adefno_vs_avg_pct:+.1f}% (above the year's own average)

### 3. IPB52300S — Defense & Space Industrial Production Index
Source: FRED series IPB52300S (https://fred.stlouisfed.org/series/IPB52300S)
Unit: Index (2017 = 100), seasonally adjusted
Interpretation: Measures how much defense and space equipment US manufacturers are
  currently producing relative to the 2017 baseline. Available to the model as an
  input feature.

Last observed value: {ipb_last_month_label}: {ipb_last_val:.2f}
12-month trend 2025: Jan 2025 = {ipb_jan_val:.2f} → {ipb_last_month_label} = {ipb_last_val:.2f}
                     (change: {ipb_12m_change:+.2f} index pts, {ipb_direction} trend)

### 4. ML Forecast — US Federal Defense Market Volume 2026
Model type: Tree-based gradient boosting regressor (details not relevant for this
narrative; treat as "the forecast model").
Training data: FRED macro series 2000–2025.

How the forecast is generated:
  - For each of the 12 months in 2026, the model uses recent FDEFX history (the
    previous 1–6 months of FDEFX values, plus rolling averages) together with the
    current values of ADEFNO and IPB52300S.
  - Month-by-month recursion: each new FDEFX prediction is used as input for the
    following month's prediction (so the FDEFX-related inputs are themselves
    updated each step).
  - The external features ADEFNO and IPB52300S are held constant at their
    {adefno_last_month} / {ipb_last_month_label} values for all 12 forecast months —
    they do not update. This is a deliberate simplification documented as a model
    limitation.

Forecast unit: USD billions, annualised spending rate (same unit as FDEFX above)

Monthly forecast:
{forecast_table}

2026 forecast average: {forecast_avg_bn} USD bn/year  (range: {forecast_min_bn}–{forecast_max_bn} USD bn)
Model reference level (mean of model outputs on its 2002–2025 training data): {base_value_bn:.1f} USD bn/year.
  Note: This is a descriptive statistic of the training data, not a target value
  the model actively pulls toward. Use it only as a rough indicator of where the
  bulk of historical observations sat, not as an "equilibrium" the model seeks.

Context: The most recent actual value ({fdefx_last_quarter}) was {fdefx_last_val:,.1f} USD bn.
The forecast average of {forecast_avg_bn} USD bn is {fdefx_step_down:.1f} USD bn below this
recent level. Whether this step-down reflects a genuine pattern the model detected or
an artefact of the recursive forecasting mechanics is addressed in Section 2.

### 5. Forecast Driver Breakdown
Method: Contribution of each input feature to each monthly forecast, expressed in
USD billions relative to the model's reference level from training data ({base_value_bn:.1f}).
Averaged across all 12 forecast months.

Top-10 drivers by average absolute contribution:
{importance_table}

Cumulative contribution of the three largest drivers (fdefx_lag_1/2/3):
  {lag1_contrib:.2f} + {lag2_contrib:.2f} + {lag3_contrib:.2f} = {cumul_lag123:.2f} USD bn pushed above the reference level
  (this is a combined effect, not an average).

Cumulative contribution of ADEFNO and IPB52300S combined: {cumul_external:.2f} USD bn.

Feature name glossary (plain language):
  fdefx_lag_1 / lag_2 / lag_3   Recent FDEFX spending levels (1, 2, 3 months before forecast)
  fdefx_lag_4 / lag_5 / lag_6   Older FDEFX spending levels (4–6 months before forecast)
  fdefx_rolling_3m_mean         Average FDEFX level over the past 3 months
  fdefx_rolling_6m_mean         Average FDEFX level over the past 6 months
  fdefx_rolling_12m_mean        Average FDEFX level over the past 12 months
  IPB52300S                     Defense production capacity index (level)
  ADEFNO_lag_6                  New defense orders placed 6 months prior

Observation for Section 2: Features derived from FDEFX's own history (the first five
rows and two rolling averages above) dominate the forecast. External signals
(ADEFNO, IPB52300S) contribute materially less. Section 2 must address what this
implies for the controller.

---

## Analysis Request

Write exactly the four sections below, then an Executive Summary.
Do not add sections. Do not discuss market share (handled separately in SAC).
Do not add geopolitical speculation beyond what the data supports.
Use plain business language throughout — avoid technical modelling terminology.

---

### Section 1 — Defense Market Volume & Outlook

Using only the FDEFX values provided above:

- Explain in plain terms what FDEFX measures and why it is relevant for a company
  entering the US defense segment (2–3 sentences).
- **State explicitly that FDEFX is used (rather than new defense orders) because it
  measures realised expenditures, which is the invoice-equivalent quantity and
  therefore aggregates consistently with the other two building blocks (B1 and B2)
  in the SAC planning model.** One sentence, placed early in the section.
- Describe the 2026 forecast trajectory: the step-down from the {fdefx_last_quarter}
  actual of {fdefx_last_val:,.1f} USD bn to the forecast average of {forecast_avg_bn} USD bn,
  and the subsequent stabilisation in the {forecast_min_bn}–{forecast_max_bn} USD bn range.
- Place the 2026 forecast average in the context of the 2020–2025 range
  ({fdefx_min_val:,.1f}–{fdefx_max_val:,.1f} USD bn).
- Explain in plain language why the initial January 2026 value ({jan_2026_val} USD bn)
  is higher than mid-year values: it reflects the model using the last actual data
  point ({fdefx_last_quarter}: {fdefx_last_val:,.1f} USD bn) as its starting reference,
  and then gradually weighing this less as the 12-month forecast rolls forward.
- **Do not frame the model as "reverting to" or "seeking" the reference level of
  {base_value_bn:.1f} USD bn.** That number is a training-data statistic, not a target
  the model actively pulls toward. You may mention it as context for where most
  historical observations sat, but not as an equilibrium the forecast moves toward.
- Keep to 3–4 short paragraphs.

### Section 2 — What Drives the Forecast

Using only the driver table and feature glossary above:

- **Headline observation (lead with this):** The forecast is dominated by FDEFX's
  own recent spending history. The three largest drivers are all prior FDEFX values
  (lag_1, lag_2, lag_3), contributing a combined {cumul_lag123:.2f} USD bn above the
  reference level. External signals — new defense orders (ADEFNO) and the defense
  production index (IPB52300S) — contribute a combined {cumul_external:.2f} USD bn.
  In plain terms: **this is a persistence model that mostly extrapolates recent
  spending, not a model that reads policy signals from order flow or production
  capacity.**

- **What this means for the controller (be direct, not hedging):**
  (a) Forecast confidence is high when recent spending patterns are a good guide
      to the near future — i.e., in stable policy environments.
  (b) The model will be slow to capture turning points driven by budget decisions
      or large contract awards that have not yet translated into actual outlays.
  (c) The elevated year-end order book (ADEFNO {adefno_last_month} at
      {adefno_last_value:,.0f} USD mn, {adefno_vs_avg_pct:.1f}% above the 2025
      average of {adefno_12m_avg:,.0f} USD mn) is a signal the model largely does
      not act on — it contributes only {adefno_contrib_val:.2f} USD bn to the
      forecast. A controller should treat this as a **qualitative upside risk** that
      sits outside the model's quantitative view.
  (d) Production capacity (IPB52300S) rose over 2025
      ({ipb_jan_val:.2f} → {ipb_last_val:.2f}) and contributes {ipb_contrib_val:.2f}
      USD bn — a mild positive, but again small relative to the persistence drivers.

- Keep this section tight: roughly one paragraph for the headline observation,
  one paragraph for the controller implications. No invented numbers. Do not
  soften the persistence-dominance finding — it is the single most important
  thing the controller needs to understand about this model.

### Section 3 — Quarterly Monitoring KPIs

Provide a table of exactly 4 KPIs the controller should track each quarter to validate
or revise the market volume assumption. For each KPI:
  - Name and plain-language description
  - Official primary source with exact URL
  - Update cadence (monthly / quarterly)
  - What to watch for (one sentence — no invented thresholds)

Use Markdown table format.

### Section 4 — Risks and Forecast Limitations

Cover exactly two topics:

(a) **US Federal Budget Risk**: Explain in plain language how continuing resolutions or
    a debt-ceiling impasse could affect actual US defense spending in 2026.
    Do not invent probability estimates.
    If you reference budget figures, cite CBO or DoD Comptroller with a URL.

(b) **Frozen External Inputs Limitation**: Explain in terms a controller understands
    what it means that ADEFNO and IPB52300S are held fixed at their {adefno_last_month} /
    {ipb_last_month_label} values for all 12 forecast months, while FDEFX's own recent
    history updates recursively.
    The forecast is **not progressively less reliable over time in a mechanical sense**
    — it is *event-sensitive*: reliability breaks down when ADEFNO or IPB52300S in
    reality diverges materially from its {adefno_last_month} value, regardless of
    whether that happens in February or November. Identify what kind of real-world
    event would most plausibly cause such divergence (e.g., a major new contract
    award, a production disruption, a supplemental appropriation).

---

### Executive Summary

Provide exactly 6 bullet points. Each bullet **must be no longer than 25 words**.
Count words before finalising. Written for direct use in a SAC Planning Story or
management presentation slide.

Bullets must cover, in this order:
  1. Core forecast message (2026 average, comparison to 2025 level)
  2. The persistence-dominance of the model and what it means for confidence
  3. Single most important KPI to monitor with update cadence
  4. Primary budget risk to the forecast
  5. What would most plausibly invalidate the forecast (event-sensitive, not
     time-decaying)
  6. Recommendation: use a scenario range, not a point estimate
"""

# ---------------------------------------------------------------------------
# Print full prompt before API call
# ---------------------------------------------------------------------------
SEP = '=' * 72
print(SEP)
print('SYSTEM PROMPT')
print(SEP)
print(SYSTEM_PROMPT)
print()
print(SEP)
print('USER PROMPT')
print(SEP)
print(USER_PROMPT)
print(SEP)
print(f'Model:         {MODEL}')
print(f'Max tokens:    {MAX_TOKENS}')
print(f'Prompt length: {len(USER_PROMPT):,} characters')
print(SEP)
print()

# ---------------------------------------------------------------------------
# API call (streaming)
# ---------------------------------------------------------------------------
print(f'Calling {MODEL} ...\n')
print('-' * 72)

llm_response = ''
with client.messages.stream(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    system=SYSTEM_PROMPT,
    messages=[{'role': 'user', 'content': USER_PROMPT}],
) as stream:
    for text in stream.text_stream:
        llm_response += text
        print(text, end='', flush=True)

print('\n' + '-' * 72)
print('Streaming complete.')

SYSTEM PROMPT
You are a senior macroeconomic analyst. Your audience is a corporate controller at an
industrial B2B company — not a data scientist. Write in clear business language.
Avoid technical modelling terms such as "XGBoost", "SHAP", "autoregressive", or
"time-series split". Instead use plain equivalents:
  - "the forecast model" (not "XGBoost")
  - "what the model relied on most" or "main forecast driver" (not "SHAP")
  - "recent spending history" (not "autoregressive lags")
  - "spending rate" or "annualised spending level" (not "SAAR")
  - "the model's reference level from training data" (not "long-run average" or "baseline")
Prioritize verifiable sourcing over narrative flourish. Brevity is a feature.

Be intellectually honest: if the data shows an inconvenient pattern (e.g., that a
supposedly important feature contributes little), surface it rather than smoothing
it over. A controller needs to know where the model is strong and where it is weak.

USER PROMPT
## Sourcing Rule

In [ ]:
# Render response as formatted Markdown in the notebook
display(Markdown(llm_response))

# Save to data/processed (consistent with other processed outputs in this pipeline)
output_path = Path(DATA_PROCESSED) / 'llm_defense_analysis_2026.md'
with open(output_path, 'w', encoding='utf-8') as f:
    f.write('# LLM Macroeconomic Analysis — US Defense Market 2026\n\n')
    f.write(f'**Model:** {MODEL}  \n')
    f.write(f'**Prompt length:** {len(USER_PROMPT):,} characters  \n\n')
    f.write('---\n\n')
    f.write(llm_response)

print(f'Response saved to: {output_path}')